# 00. Google Colab & Gemini 사용 가이드


> **시나리오 안내** — 이 실습은 가상의 이커머스 마켓플레이스 **쇼핑몰**(자체 PB 라인: 베이직 · 프리미엄 · 라이트 · 한정판)를 소재로 합니다.

| 항목 | 내용 |
|------|------|
| 대상 | Colab을 처음 쓰거나 익숙하지 않은 수강생 |
| 목적 | 01~06 모든 실습의 **공통 기반** — Colab 기본 조작 + Gemini AI 활용법 |
| 사전 준비 | Google 계정 (Gmail) |
| 소요 시간 | ~15분 |

> **이 노트북을 먼저 실행한 뒤 01~06 실습을 진행하세요.** 이후 실습에서 "런타임 변경", "GPU 확인", "pip install" 같은 용어가 별도 설명 없이 등장합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## 1. Google Colab이란?

**Google Colab(Colaboratory)**은 브라우저에서 Python 코드를 실행할 수 있는 **Google의 무료 클라우드 노트북 환경**입니다.

**왜 Colab을 쓰는가?**

- **설치가 필요 없습니다.** 브라우저만 있으면 됩니다. Python, PyTorch 등이 이미 설치되어 있습니다. GPU 런타임을 선택하면 CUDA 가속도 자동으로 활성화됩니다.
- **GPU를 무료로 쓸 수 있습니다.** NVIDIA T4 GPU(16GB VRAM)를 무료로 할당받을 수 있습니다. 02 실습(LoRA 학습)에서 이 GPU를 사용합니다.
- **Google Drive와 연동됩니다.** 학습 데이터, 모델 체크포인트를 Drive에 저장하면 세션이 끊겨도 파일이 보존됩니다.

### Colab 노트북의 구조

Colab 노트북은 **셀(Cell)**로 이루어져 있습니다. 셀은 두 종류입니다.

- **코드 셀**: Python 코드를 작성하고 실행하는 칸. 실행하면 결과가 바로 아래에 출력됩니다.
- **텍스트 셀 (마크다운 셀)**: 설명, 메모, 제목을 적는 칸. 코드가 아니라 글을 쓰는 용도입니다.

실습 노트북은 "텍스트 셀(설명) → 코드 셀(실행) → 텍스트 셀(설명) → 코드 셀(실행)..." 이 반복되는 구조입니다.

---
## 2. Colab 기본 조작

### 2-1. 노트북 열기

1. [colab.research.google.com](https://colab.research.google.com) 접속
2. 강사가 공유한 노트북 링크를 클릭하거나, `파일 → 노트북 열기`에서 Google Drive/GitHub URL 입력
3. **"사본 저장" (★ 필수)**: `파일 → 드라이브에 사본 저장`을 눌러 **내 Drive에 복사**해야 수정/실행이 가능합니다.

### 2-2. 셀 실행

| 방법 | 설명 |
|------|------|
| **Shift + Enter** | 현재 셀 실행 후 다음 셀로 이동 (가장 많이 씀) |
| **Ctrl + Enter** | 현재 셀만 실행 (커서 이동 안 함) |
| 셀 왼쪽 ▶ 버튼 클릭 | 마우스로 실행 |
| `런타임 → 모두 실행` | 전체 셀을 위에서부터 순서대로 실행 |
| `런타임 → 이전 실행 중단` | 오래 걸리는 셀을 강제 중지 |

**👇 아래 셀을 Shift+Enter로 실행해보세요.**

In [ ]:
# 셀 실행 연습 — Shift+Enter를 눌러 실행하세요
print("첫 번째 셀 실행 성공! ✅")
print("Python이 이미 설치되어 있습니다.")

### 2-3. 런타임(Runtime) 이해

**런타임**은 코드가 실행되는 **서버 환경**입니다. Colab이 Google 서버에 가상 컴퓨터를 하나 할당해주는 것이라고 생각하면 됩니다.

| 런타임 | 하드웨어 | 용도 | 실습에서의 사용 |
|--------|----------|------|----------------|
| **CPU** | CPU만 사용 | 가벼운 전처리, GGUF 추론 | 04(양자화 추론), 05(Gradio) |
| **T4 GPU** | NVIDIA T4 (16GB VRAM) | 모델 학습, FP16 추론 | 02(SFT 학습), 03(평가) |

**런타임 변경 방법:**
1. 메뉴에서 `런타임 → 런타임 유형 변경`
2. `하드웨어 가속기`를 선택 (None = CPU, T4 GPU, 등)
3. `저장` 클릭

> ⚠️ **런타임을 변경하면 현재 세션이 초기화됩니다.** 이미 실행한 셀의 변수, 다운로드한 파일이 사라집니다. 단, Google Drive에 마운트한 파일은 보존됩니다.

### 2-4. GPU 사용량 확인

T4 GPU 런타임에서 GPU 메모리를 얼마나 쓰고 있는지 확인하는 방법입니다.

- `Memory-Usage`가 16GB에 가까워지면 **메모리 부족(OOM)**으로 에러가 납니다.
- 02 실습에서 "7B 모델 FP16 = 14GB" 공식이 여기서 현실이 됩니다.

> ⚠️ 현재 CPU 런타임이면 아래 셀에서 에러가 납니다. T4 GPU 런타임으로 변경 후 실행하세요.

In [ ]:
# GPU 상태 확인 — T4 GPU 런타임에서 실행하세요
!nvidia-smi

### 2-5. 패키지 설치 (pip install)

Colab에는 기본 패키지(numpy, pandas, torch 등)가 이미 설치되어 있지만, 실습에서 쓰는 추가 라이브러리는 직접 설치해야 합니다.

- `!`를 앞에 붙이면 코드 셀에서 **터미널 명령어**를 실행할 수 있습니다.
- 설치는 **현재 세션에서만 유효**합니다. 세션이 끊기면 다시 설치해야 합니다.
- 그래서 모든 실습 노트북의 **셀 1**이 `pip install ...`로 시작합니다.

In [ ]:
# pip install 연습 — 가벼운 패키지로 설치 방법을 확인합니다
# 실제 실습에서는 unsloth, langfuse, gradio 등을 설치합니다
!pip install --quiet requests
print("설치 완료 ✅")

### 2-6. Google Drive 연동

학습 데이터나 모델 체크포인트를 세션이 끊겨도 보존하려면 Google Drive를 마운트합니다.

실행하면 Google 계정 인증 팝업이 뜨고, 승인하면 `/content/drive/MyDrive/`에서 내 Drive 파일에 접근할 수 있습니다.

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# 마운트 확인
print("Drive 마운트 완료 ✅")
print("내 Drive 경로:", "/content/drive/MyDrive/")

In [ ]:
# /content 디렉토리의 내용을 나열하여 'drive' 폴더가 있는지 확인합니다.
# 'drive' 폴더가 보인다면 마운트가 성공적으로 된 것입니다.
!ls /content/

# 또한, 내 Drive 파일은 /content/drive/MyDrive/ 경로에서 접근할 수 있습니다.
# 이 경로에 어떤 파일이 있는지 확인하여 마운트 여부를 다시 한번 확인해볼 수 있습니다.
print("\n/content/drive/MyDrive/ 폴더 내용:")
!ls /content/drive/MyDrive/

### 2-7. 세션 제한 + 끊겼을 때 대처

| 제한 사항 | 내용 |
|-----------|------|
| **세션 시간** | 무료 기준 **최대 ~90분** 연속 사용. 사용하지 않으면 더 빨리 끊김 |
| **GPU 할당량** | 하루에 사용 가능한 GPU 시간이 제한적. 많이 쓰면 일시적으로 GPU 할당 불가 |
| **세션 끊기면?** | 메모리의 모든 변수, 다운로드한 파일이 사라짐. Google Drive에 저장한 것만 남음 |

**대처법:**

1. **중요 파일은 Drive에 저장.** 학습된 어댑터, 결과 파일 등을 `/content/drive/MyDrive/`에 저장해두면 세션이 끊겨도 안전합니다.
2. **HF Hub에 push.** 02 실습에서 학습한 모델을 HF Hub에 올려두면, 03/04/05 실습에서 어디서든 다시 받을 수 있습니다.
3. **셀 1부터 다시 실행.** 세션이 끊기면 `런타임 → 모두 실행`으로 처음부터 재실행합니다. 데이터/모델이 Drive나 HF Hub에 있으면 몇 분이면 복구됩니다.
4. 우리의 실습자료는 이미 이렇게 하도록 설계되어 있습니다.

---
## 3. Gemini Native 활용법

### 3-1. Gemini Native란?

**Gemini**는 Google이 만든 AI 모델이고, Colab에는 이 Gemini가 **코딩 어시스턴트로 내장**되어 있습니다.

별도 설치나 API 키가 필요 없습니다. Colab을 쓰는 것만으로 자동으로 사용 가능합니다.

### 3-2. 핵심 기능 4가지

#### 1) 코드 자동 완성

코드를 입력하다가 멈추면, Gemini가 **다음에 올 코드를 회색으로 제안**합니다.

- **Tab**을 누르면 제안을 수락합니다.

**👇 아래 셀에서 `from transformers`까지 타이핑하면 Gemini가 자동 완성을 제안합니다.**

In [ ]:
# 자동 완성 연습
# 'from transformers' 까지 직접 타이핑해보세요
# Gemini가 'from transformers import Auto', 'from transformers import pipeline' 등을 회색으로 제안합니다
# Tab 키로 수락할 수 있습니다

#### 2) 코드 설명 요청

실습 셀에 있는 코드가 이해가 안 될 때:

- 코드 셀 위에 마우스를 올리면 셀 오른쪽에 **"코드 설명"** 버튼이 나타납니다 → 클릭하면 Gemini가 자동으로 코드를 설명해줍니다
- 또는 화면 **우측 패널**의 **Gemini 탭**에서 직접 질문을 입력할 수 있습니다

> 우측 패널이 보이지 않으면 상단 메뉴 `도구 → Gemini`를 클릭하세요.

**👇 아래 셀의 코드가 뭘 하는지 "코드 설명" 버튼을 눌러 확인해보세요.**

In [ ]:
# 이 코드가 뭘 하는지 Gemini에게 설명을 요청해보세요
# 셀 위에 마우스를 올리면 오른쪽에 "코드 설명" 버튼이 나타납니다

lora_config = {
    "r": 8,
    "target_modules": ["q_proj", "v_proj"],
    "lora_alpha": 16,
    "lora_dropout": 0.05,
}

print("LoRA 설정:", lora_config)

#### 3) 에러 디버깅

코드 셀을 실행했는데 **빨간 에러 메시지**가 나왔을 때:

- 에러 메시지 아래에 **"Gemini에게 이 에러 설명 요청"** 버튼이 자동으로 나타납니다.

**👇 아래 셀을 실행하면 에러가 납니다. 에러 메시지 아래 버튼으로 Gemini에게 원인을 물어보세요.**

In [ ]:
# 에러 디버깅 연습 — 이 셀은 의도적으로 에러를 냅니다
# 실행 후 에러 메시지 아래 'Gemini에게 설명 요청' 버튼을 클릭해보세요

print(undefined_variable)  # NameError 발생

#### 4) 코드 생성 요청

빈 코드 셀에서 자연어로 **"이런 코드를 작성해줘"**라고 입력할 수 있습니다.

- 우측 패널의 **Gemini 탭**에서 프롬프트 입력
- 예: `"matplotlib으로 막대그래프를 그려줘. x축은 모델 이름, y축은 점수."`

> ⚠️ **Gemini가 생성한 코드는 반드시 검토하세요.** 라이브러리 버전이 다르거나 실습 환경에 맞지 않는 코드를 생성할 수 있습니다.

**👇 아래 빈 셀에 붙여넣어서 실행해보세요.**

In [ ]:
# 코드 생성 연습 — 빈 셀에서 ✨ 아이콘을 눌러보세요
# 예: "matplotlib으로 모델별 점수를 막대그래프로 그려줘"

### 3-3. Gemini 활용 팁

| 상황 | Gemini 활용법 |
|------|---------------|
| 코드가 이해 안 됨 | 셀 위에 마우스 올리기 → **"코드 설명"** 버튼 클릭 |
| 에러가 남 | 에러 메시지 아래 **"Gemini에게 설명 요청"** 클릭 |
| 결과를 시각화하고 싶음 | 우측 패널 Gemini 탭 → "위 결과를 막대그래프로 그려줘" 입력 |
| 코드를 수정하고 싶은데 어떻게 바꿔야 할지 모르겠음 | 코드 선택 → "이 코드에서 batch_size를 4로 바꾸려면?" 질문 |
| 영어 에러 메시지 해석 | 에러 메시지 복사 → Gemini에게 "이 에러가 뭐야?" 질문 |

---
## 4. 자주 묻는 질문

### Q: GPU가 할당 안 돼요 ("GPU quota exhausted")

하루 GPU 사용량을 초과했을 때 발생합니다. **해결:**
- 몇 시간 뒤에 다시 시도하면 할당량이 복구되지만 또 다른 Google 계정을 생성해서 새로 할당받는 것을 추천합니다.
- 또는 `런타임 → 런타임 유형 변경 → CPU`로 바꿔서, GPU가 필요 없는 셀부터 먼저 진행합니다.
- 04/05 실습은 CPU 런타임만으로도 가능합니다.

### Q: 세션이 자꾸 끊겨요

- 브라우저 탭을 **활성 상태로 유지**하세요. 다른 탭으로 오래 넘어가면 세션이 끊길 수 있습니다.
- 학습처럼 오래 걸리는 셀이 돌아가는 동안에도 탭을 열어두세요.
- 세션이 끊겨도 **Drive/HF Hub에 저장한 것은 안전**합니다.

### Q: `!pip install` 했는데 "모듈을 찾을 수 없습니다"가 뜨면?

설치 후 **런타임을 재시작**해야 하는 경우가 있습니다.
- `런타임 → 런타임 다시 시작`을 누른 뒤, `pip install` 셀을 제외하고 그 아래 셀부터 재실행하세요.

### Q: Gemini가 한국어로 안 답해요

- Gemini에게 질문할 때 **한국어로 입력**하면 대부분 한국어로 답합니다.
- 간혹 영어로 답하면 "한국어로 다시 설명해줘"를 추가하면 됩니다.

---
## 5. 다음 실습

이 노트북을 완료했으면 아래 순서로 진행하세요.

| 실습 | 파일 | 런타임 | 내용 |
|------|------|--------|------|
| 01 | `01_synthetic_data.ipynb` | CPU | 합성 데이터 생성 |
| 02 | `02_sft_train.ipynb` | **T4 GPU** | LoRA SFT 학습 |
| 03 | `03_sft_eval.ipynb` | T4 GPU | 4조건 평가 + Langfuse |
| 04 | `04_quantization.ipynb` | GPU→CPU 전환 | GGUF 양자화 |
| 05 | `05_gradio_deploy.ipynb` | **CPU** | Gradio 데모 배포 |
| 06 (옵션) | `06_agentic_rag.ipynb` | CPU | Agentic RAG |

> **Colab은 브라우저에서 Python + GPU를 무료로 쓸 수 있는 환경이고, 내장된 Gemini에게 코드 설명/에러 디버깅/코드 생성을 바로 요청할 수 있다. 01~06 모든 실습의 공통 기반이므로 런타임 종류, 세션 제한, pip install, Drive 연동을 미리 익혀두자.**